# Fine-tune Lumen on custom microscopy data

This notebook shows the core API for a shared-backbone, multi-head microscopy model with classification, segmentation, contrastive, and MAE losses. Replace the synthetic tensors with your microscopy dataloader batches shaped like `{'image', 'label', 'mask', 'unlabeled'}`.

In [ ]:
import torch

from lumen.models import build_encoder
from lumen.training import MultiHeadMicroscopyModel, MultiHeadMicroscopyTrainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = build_encoder('eupe', embed_dim=128, depth=2, num_heads=4).to(device)

model = MultiHeadMicroscopyModel.with_default_heads(
    encoder,
    num_classes=3,                 # cell type / organelle class
    num_segmentation_classes=2,    # foreground / background, nuclei, mitochondria, etc.
    use_contrastive=True,
    use_mae=True,
    mask_ratio=0.5,
).to(device)

trainer = MultiHeadMicroscopyTrainer(
    model,
    loss_weights={
        'classification': 1.0,
        'segmentation': 1.0,
        'contrastive': 0.2,
        'mae': 0.5,
    },
    stop_gradient_heads=set(),
    lr=1e-4,
).to(device)

batch = {
    'image': torch.randn(2, 1, 64, 64, device=device),
    'label': torch.tensor([0, 2], device=device),
    'mask': torch.randint(0, 2, (2, 64, 64), device=device),
    'unlabeled': torch.randn(2, 1, 64, 64, device=device),
}
metrics = trainer.train_step(batch)
metrics


For few-shot fine-tuning, disable SSL heads or lower their weights, freeze the encoder with the staged trainer utilities, and monitor `top1_accuracy`, `macro_f1_score`, and `mean_iou` from `lumen.training.eval`.